In [ ]:
""" 対象レースのコメントを取る """
from datetime import datetime
from time import sleep
import re
import sys
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import warnings
warnings.filterwarnings('ignore')
import pyperclip
import ipywidgets as widgets
from IPython.display import HTML
import myrace as mrc
from mycolor import *

""" 対象のレース """
def get_RaceInfomation(race_code):
    driver = webdriver.PhantomJS()
    race_url = "http://race.netkeiba.com/?pid=race_old&id=c2019" + race_code
    driver.get(race_url)

    read_xpath = "//table[@class='race_table_old nk_tb_common']"
    for _ in range(3):
        try:
            WebDriverWait(driver, 30).until(EC.presence_of_element_located((By.XPATH, read_xpath)))
            sleep(3)
        except TimeoutException as te:
            print("driver waiting..", te)
        else:
            break
    else:
            driver.close()
            sys.exit()
    
    return driver

rc = mrc.Rcd('東京sun')
race_code = rc.grc() + '11'
driver = get_RaceInfomation( race_code )
# print(driver.page_source)
print(driver.title[:-26], "Code loading ", datetime.now().strftime("%Y/%m/%d %H:%M:%S"))

In [ ]:
""" netkeiba出馬表を別ウィンドウでpopup """

link1 = '<a href="https://race.netkeiba.com/?pid=race_old&id=c2019'
link2 = race_code
link3 = '&mode=top" style="font-size: 12pt;" onclick="navigateTargetUrl(); return false;">'
link4 = driver.title[:-26]
link5 = '</a>'
link6 = '''
<script>
  // a タグの href 属性に記述された URL を、新規ウィンドウで開く関数
  function navigateTargetUrl() {
    window.open(this.event.target.href, null, 'width=900, height=900');
  }
</script>
'''
PopUp_script = link1 + link2 + link3 + link4 + link5 + link6
HTML(PopUp_script)

In [ ]:
horse_driver = webdriver.PhantomJS()
bbs_driver = webdriver.PhantomJS()

def forced_termination():
    horse_driver.close
    bbs_driver.close
    print("Error occurred..")
    sys.exit()

""" race title """
race_title = re.sub("出馬表 ", "", driver.title[:-26])
race_title = re.sub("\|", "", race_title)
race_corse = driver.find_elements_by_xpath("//dl[@class='racedata fc']/dd/p")[0].text
race_condition = driver.find_elements_by_xpath("//dl[@class='racedata fc']/dd/p")[1].text
print(race_title + " " + race_corse + " " + race_condition + "\n")

""" horse list """
no_lst = driver.find_elements_by_xpath("//td[@class='umaban']")
horse_lst = driver.find_elements_by_xpath("//td[@class='txt_l horsename']")
age_df = pd.io.html.read_html(driver.page_source)
age_lst = [age_df[0]['性齢'].values[s][0] for s in range(len(horse_lst))]
link_lst = driver.find_elements_by_xpath("//td[@class='txt_l horsename']/div/a")
odds_lst = driver.find_elements_by_xpath("//td[@class='txt_r']")
pop_lst = driver.find_elements_by_xpath("//td[contains(@class, 'ml')]") #popularity
jockey_lst = driver.find_elements_by_xpath("//td[@class='txt_l']/a[contains(@href, '/jockey/')]")

""" element for webdriver """
bbs_title_xpath = "//p[@class='db_prof_top_bbs_title png_bg']" #掲示板(xx件)
seealot_xpath = "//dd[@class='DB_ProfHead_dd_01 fc']/p[@class='detail_link']/a" #もっと見る

total_comments_lst = []
horse_url_lst = []
check_lst = [[] for i in range(len(horse_lst))]
save_nickname_lst = [[] for i in range(len(horse_lst))]
save_comment_lst = [[] for i in range(len(horse_lst))]

""" main comment loop """
for h in range(len(horse_lst)):
    """ 対象の馬 """
    print("\n")
    print(Color.Blue + str(no_lst[h].text).rjust(2) + " " + horse_lst[h].text + Color.End, end=" ")
    print(Color.Cyan + "性齢:" + age_lst[h] + Color.End, end=" ")
    print(Color.Blue + "騎手:" + jockey_lst[h].text + Color.End, end=" ")
    print(Color.Cyan + "ODDS:" + odds_lst[h].text + Color.End, end=" ")
    print(Color.Cyan + "人気:" + str(pop_lst[h].text).rjust(2) + Color.End)
    
    horse_url = link_lst[h].get_attribute("href")
    horse_url_lst.append(horse_url)
    horse_driver.get(horse_url)
    
    for _ in range(3):
        try:
            WebDriverWait(horse_driver, 30).until(EC.presence_of_element_located((By.XPATH, bbs_title_xpath)))
        except TimeoutException as te:
            print("horse_driver can't read a result table", te)
        else:
            break
    else:
        forced_termination()
            
    for _ in range(3):
        try:
            WebDriverWait(horse_driver, 30).until(EC.presence_of_element_located((By.XPATH, seealot_xpath)))
        except TimeoutException as te:
            print("horse_driver can't read a seealot link", te)
        else:
            break
    else:
        forced_termination()
    
    sleep(10)
    horse_df = pd.io.html.read_html(horse_driver.page_source)
    """ 血統、賞金、通算、主な勝鞍 """
    blood = "父:" + horse_df[2][0][0] + " / 母:" + horse_df[2][0][2]
    for r in range(len(horse_df[1])):
        if(horse_df[1][0][r] == '生年月日'):
            birthday = horse_df[1][1][r]
        if(horse_df[1][0][r] == '調教師'):
            trainer = horse_df[1][1][r]
        if(horse_df[1][0][r] == '獲得賞金'):
            prize = horse_df[1][1][r]
        elif(horse_df[1][0][r] == '通算成績'):
            total_rlt = horse_df[1][1][r]
        elif(horse_df[1][0][r] == '主な勝鞍'):
            victory = horse_df[1][1][r]
    print("  {} {}(生) 調教師:{}".format(blood, birthday, trainer))
    print("  賞金:{} 成績:{} 主な勝鞍:{}".format(prize, total_rlt, victory))
    
    """ past 5 runs """
    if(len(horse_df) <4 ):
        print("a table of past race is not exist..")
    else:
        drop_lst = ["映像", "枠番", "馬場指数", "ﾀｲﾑ指数", "ペース", "厩舎ｺﾒﾝﾄ", "備考", "勝ち馬(2着馬)", "賞金"]
        result_df = horse_df[3].drop(drop_lst, axis=1)
        if(len(result_df)<6):
            display(result_df)
        else:
            display(result_df[:5])
    
    """ bbs(xx件) """
    total_comments = horse_driver.find_element_by_xpath(bbs_title_xpath) #コメント数
    total_comments_lst.append(total_comments.text)
    bbs_link = horse_driver.find_element_by_xpath(seealot_xpath)
    bbs_url = bbs_link.get_attribute("href")
    print(" " + Color.Magenta + total_comments_lst[h] + Color.End)
    print(bbs_url)

    bbs_driver.get(bbs_url)
            
    for _ in range(3):
        try:
            WebDriverWait(bbs_driver, 30).until(EC.presence_of_element_located((By.CLASS_NAME, "user_report_list")))
        except TimeoutException as te:
            print("bbs_driver: can't read user comment", te)
        else:
            break
    else:
        forced_termination()
    
    sleep(5)
    """ bbs comment """
    nickname_obj = bbs_driver.find_elements_by_class_name('nickname')
    nickname_lst = [nickname_obj[n].text for n in range(len(nickname_obj))][::-1]
    comment_obj = bbs_driver.find_elements_by_xpath("//p[@class='comment']")
    comment_lst = [comment_obj[c].text for c in range(len(comment_obj))][::-1]
    post_time_obj = bbs_driver.find_elements_by_class_name('time_data_01')
    post_time_lst = [post_time_obj[p].text for p in range(len(post_time_obj))][::-1]
    
    #if(len(nickname_lst)==0):
    #    print("can't get nickname .. ")
    #    forced_termination()
    """ bbs comment loop """
    for n in range(len(nickname_lst)):
        print(Color.Blue + nickname_lst[n] + Color.End + " / " + Color.Cyan + post_time_lst[n] + Color.End)
        print(Color.Green + comment_lst[n] + Color.End)
        check_lst[h].append(widgets.Checkbox(description=''))
        display(check_lst[h][n])
        #pyperclip.copy(comment_lst[n].text)
        #pick = input("pickup?")
        save_nickname_lst[h].append(nickname_lst[n] + " / " + post_time_lst[n])
        save_comment_lst[h].append(comment_lst[n])
    
print("\n", "Code loading ", datetime.now().strftime("%Y/%m/%d %H:%M:%S"))

In [ ]:
print(race_title + " " + race_corse + " " + race_condition + "\n")
race_url = "https://race.netkeiba.com/?pid=race_old&id=c2019"+ race_code
link_href = '<a href='
link_url = race_url
link_opt = ' style="font-size: 12pt;" onclick="navigateTargetUrl(); return false;">'
link_a   = '</a>'
javascript = """
<script>
    function navigateTargetUrl() {
    window.open(this.event.target.href, null, 'width=900, height=900');
  }
</script>
"""
race_toppage = link_href+ link_url + link_opt + link_url + link_a +  javascript
display(HTML(race_toppage))

dropdown_lst = []
for h in range(len(horse_lst)):
    """ 対象の馬 """
    print("\n")
    print(Color.Blue + str(no_lst[h].text).rjust(2) + " " + horse_lst[h].text + Color.End, end=" ")
    print(Color.Cyan + "性齢:" + age_lst[h] + Color.End, end=" ")
    print(Color.Blue + "騎手:" + jockey_lst[h].text + Color.End, end=" ")
    print(Color.Cyan + "ODDS:" + odds_lst[h].text + Color.End, end=" ")
    print(Color.Cyan + "人気:" + str(pop_lst[h].text).rjust(2) + Color.End)
    print(" 賞金:{} 成績:{} 主な勝鞍:{}".format(prize, total_rlt, victory))
    #print(Color.PURPLE + total_comments_lst[h].text + Color.END)
    print(horse_url_lst[h])
    dropdown_lst.append(widgets.Dropdown(
        options=['◎', '○', '▲', '△', 'X', '..'],
        value='..', 
        description='予想:', 
        disabled=False))
    display(dropdown_lst[h])

    for n in range(len(check_lst[h])):
        if(check_lst[h][n].value == True):
            print(Color.Cyan + save_nickname_lst[h][n] + Color.End)
            print(Color.Green + str(save_comment_lst[h][n]) + Color.End)

In [ ]:
for i in range(len(horse_lst)):
    print(dropdown_lst[i].value, end=" ")
    print(horse_lst[i].text)

In [ ]:
a = [[] for _ in range(3)]
a[0].append(Checkbox(description='pickup?'))
a[0][0]

In [ ]:
import ipywidgets as widgets
widgets.Button(
    description='Click me',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    #tooltip='Click me',
    icon='check'
)

In [ ]:
widgets.Checkbox(
    value=False,
    description='', #'Check me',
    disabled=False,
    icon='check'
)
#check_lst[h].append(Checkbox(description=''))
#display(check_lst[h][n])

In [ ]:
widgets.SelectMultiple(
    options=['Apples', 'Oranges', 'Pears'],
    value=['Oranges'],
    #rows=10,
    description='Fruits',
    disabled=False
)